# NB01b – Daten Simulation
### CAS Information Engineering – Scripting Project (Dev/Helper)

---
Erzeugt simulierte Rohdaten unter `sim/raw/` im **identischen Format wie NB01**.

**Nicht Teil der Einreichung** — nur Entwicklungshilfsmittel bis echte Daten verfügbar.  
Kann mit echten Daten mit `MODE = 'sim'` in NB02 verwendet werden.

---
[↑ Projektübersicht](00_Project_Overview.ipynb)


In [ ]:
# ── Projektstruktur ────────────────────────────────────────────────────────────
import os, sys, warnings, datetime
import numpy  as np
import pandas as pd
warnings.filterwarnings('ignore')

# Datenquelle: 'data' = echte Downloads | 'sim' = simuliert
MODE = 'data'   # ← hier anpassen: 'data' | 'sim'

DIR_RAW          = os.path.join(MODE, 'raw')
DIR_PROCESSED    = os.path.join(MODE, 'processed')
DIR_INTERMEDIATE = os.path.join(MODE, 'intermediate')
DIR_CHARTS       = os.path.join('output', 'charts', 'sim')  # SIM-Charts
DATAINDEX        = 'dataindex.csv'

# SIM-Mode: nur sim/ Ordner anlegen; data/ wird von NB01 angelegt
for d in [DIR_RAW, DIR_PROCESSED, DIR_INTERMEDIATE, DIR_CHARTS,
          os.path.join('sim', 'raw'), os.path.join('sim', 'processed'),
          os.path.join('sim', 'intermediate'),
          os.path.join('sim', 'intermediate', 'realistisch')]:  # aktiver Szenario-Ordner
    os.makedirs(d, exist_ok=True)

print(f'MODE         : {MODE}')
print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTERMEDIATE)}')

In [ ]:
# ── dataindex.csv Helper ───────────────────────────────────────────────────────
def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note=''):
    """
    Zentrales Datenquellen-Register. Jeder Eintrag bleibt als Historie erhalten.
    status: active | superseded | error | partial
    """
    ts = datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
    if os.path.exists(DATAINDEX):
        df_idx = pd.read_csv(DATAINDEX)
        mask   = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status']        = 'superseded'
            df_idx.loc[mask, 'superseded_at'] = ts
    else:
        df_idx = pd.DataFrame(columns=['timestamp','filename','source_url','local_path',
                                        'data_type','rows','size_kb','status','superseded_at','note'])
    row = {'timestamp': ts, 'filename': filename, 'source_url': source_url,
           'local_path': local_path, 'data_type': data_type, 'rows': rows,
           'size_kb': round(size_kb,1) if size_kb else None,
           'status': status, 'superseded_at': '', 'note': note}
    pd.concat([df_idx, pd.DataFrame([row])], ignore_index=True).to_csv(DATAINDEX, index=False)
    print(f'  dataindex: {filename} [{status}]')

def log_missing(source, reason):
    ts = datetime.datetime.utcnow().isoformat(timespec='seconds')
    with open(os.path.join('data','missing.txt'), 'a') as f:
        f.write(f'[{ts}] MISSING {source}: {reason}\n')
    log_dataindex(os.path.basename(source), source, '', 'raw', status='error', note=reason)
    print(f'  missing.txt: {reason}')

print('dataindex Helper bereit.')


In [ ]:
# sim/raw/ als Ziel setzen
MODE          = 'sim'
DIR_RAW       = os.path.join(MODE, 'raw')
DIR_PROCESSED = os.path.join(MODE, 'processed')
DIR_INTERMEDIATE = os.path.join(MODE, 'intermediate')
os.makedirs(DIR_RAW, exist_ok=True)
print(f'Speicherort: {os.path.abspath(DIR_RAW)}')


In [ ]:
# ── Simulation 1: ENTSO-E Spot-Preise ─────────────────────────────────────────
"""
Modell (CH-Profil empirisch aus Markdaten abgeleitet):
  Basis 60 EUR/MWh · Nachttief -20 (02-05h) · Morgenspitze +25 (07-09h)
  Solarmittagstief -10 (12-14h) · Abendspitze +30 (17-20h)
  Saisonalität ±15 · Wochenende -8 · Rauschen σ=8 · Spikes 0.5%
"""
PRICES_FILE = os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv')

np.random.seed(42)
idx = pd.date_range('2023-01-01','2024-12-31 23:00', freq='1h', tz='UTC')
h, m, dow = idx.hour, idx.month, idx.dayofweek

prices = np.clip(
    60 + (-20*np.exp(-((h-4)**2)/2) + 25*np.exp(-((h-8.5)**2)/3)
          - 10*np.exp(-((h-14)**2)/4) + 30*np.exp(-((h-19)**2)/3))
    + 15*np.cos((m-1)*2*np.pi/12)
    + np.where(dow>=5,-8,0)
    + np.random.normal(0,8,len(idx))
    + np.where(np.random.rand(len(idx))<0.005,
               np.random.choice([-50,-30,150,200],len(idx)),0),
    -60, 400)

df_sim = pd.DataFrame({'timestamp': idx.strftime('%Y-%m-%dT%H:%M:%SZ'),
                        'price_eur_mwh': prices.round(2)})
df_sim.to_csv(PRICES_FILE, index=False, encoding='utf-8')
kb = os.path.getsize(PRICES_FILE)/1024
log_dataindex('ch_spot_prices_raw.csv','SIMULATED', PRICES_FILE,'raw/sim',
              rows=len(df_sim), size_kb=kb, note='Gauss-Modell seed=42')
print(f'Preise: {len(df_sim):,} h | {prices.min():.0f}–{prices.max():.0f} EUR/MWh | {kb:.0f} KB')


In [ ]:
# ── Verifikation: Simulierte Preise ─────────────────────────────────────────
print(f'Shape : {df_sim.shape}')
print(f'Range : {df_sim["price_eur_mwh"].min():.1f} – {df_sim["price_eur_mwh"].max():.1f} EUR/MWh')
df_sim.head(3)


In [ ]:
# ── Simulation 2: Netzlast ────────────────────────────────────────────────────
"""
Modell: Grundlast 6.5 GW + Gauss-Spitzen (Morgen/Abend)
        + Saisonalität ±1.5 GW + Wochenende -0.8 GW + Rauschen σ=0.3
"""
LOAD_FILE = os.path.join(DIR_RAW, 'ch_netzlast_raw.csv')

np.random.seed(1)
idx = pd.date_range('2023-01-01','2024-12-31 23:00', freq='1h', tz='UTC')
h, m, dow = idx.hour, idx.month, idx.dayofweek

load = np.maximum(
    6.5 + 1.8*np.exp(-((h-9)**2)/6) + 2.2*np.exp(-((h-19)**2)/5)
        - 1.5*np.exp(-((h-4)**2)/3) + 1.5*np.cos((m-1)*2*np.pi/12)
        + np.where(dow>=5,-0.8,0) + np.random.normal(0,0.3,len(idx)), 2.0)

df_load = pd.DataFrame({'timestamp': idx.strftime('%Y-%m-%dT%H:%M:%SZ'),
                         'load_gw': load.round(4)})
df_load.to_csv(LOAD_FILE, index=False, encoding='utf-8')
kb = os.path.getsize(LOAD_FILE)/1024
log_dataindex('ch_netzlast_raw.csv','SIMULATED', LOAD_FILE,'raw/sim',
              rows=len(df_load), size_kb=kb, note='Gauss-Modell seed=1')
print(f'Last  : {len(df_load):,} h | {load.min():.2f}–{load.max():.2f} GW | {kb:.0f} KB')
print('\nNB1b fertig → NB2_Daten_Analyse mit MODE=\'sim\'')


In [ ]:
# ── Verifikation: Simulierte Last ────────────────────────────────────────────
print(f'Shape : {df_load.shape}')
print(f'Range : {df_load["load_gw"].min():.2f} – {df_load["load_gw"].max():.2f} GW')
df_load.head(3)
